<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Unit Testing Guide & Script Descriptions

## Introduction: Unit Testing for Data Engineers

Unit testing is when you test small parts of your code functions, classes, or data changes on their own, away from the rest of the system. For Data Engineers building complex pipelines, unit tests are the best defence against "silent data errors" and logic mistakes.

Unlike **Integration Tests** (which check if the pipeline runs from start to finish), **Unit Tests** make sure a specific change (like `handle_types` or `impute_values`) behaves exactly as expected, even when given "messy" or unexpected data.

* **Why It Matters:**

  * **Catch Mistakes Early:** Make sure your cleaning logic handles odd cases (like empty strings or `NaNs`) before you run a full update.

  * **Safe Changes:** You can improve your Pandas code with confidence. If the test passes, your changes didn't break the business logic.

  * **Documentation:** The test cases show how your functions should handle specific inputs.

* **Key Concepts in This Project:**

  * **Fixtures (`conftest.py`):** Instead of making new DataFrames for every test, we define standard "dummy data" once and pass it into any test that needs it.

  * **Mocking:** We mimic external services (like reading a file from S3 or writing to a disk). This makes sure tests run fast and don't change real files.

  * **Assertions:** These are the checks. We confirm that the output shape is correct or that a specific column has been turned into a `float`.

## Shared Test Configuration (`conftest.py`)

This file is the core of the `pytest` suite. It defines "fixtures" setup code we can reuse and pass into any test function automatically. By centralising the creation of dummy data and settings, we ensure consistent testing conditions across all test files and cut down on repeated code.

* **Key Fixtures:**

  * **`raw_churn_df`:**

    * **Goal:** Creates a standard, small Pandas DataFrame that copies the structure of the real Telco Churn dataset.

    * **Details:** It includes "dirty" data on purpose (e.g., empty strings in number columns, `NaN` values) to make sure the cleaning logic is tested strictly in later scripts.

  * **`pipeline_params`:**

    * **Goal:** Separates tests from fixed strings. It gets column names and target variables straight from the project's `src.config`.

    * **Details:** Returns a dictionary (e.g., `{'target': 'Churn', ...}`) that allows tests to adapt if the main settings change.

  * **`empty_churn_df`:**

    * **Goal:** Provides a valid DataFrame structure that has zero rows.

    * **Details:** Used to test edge cases where the input file exists but is empty, ensuring the pipeline doesn't crash on valid but empty inputs.




In [ ]:
import pytest
import pandas as pd
import numpy as np
import os
import sys

@pytest.fixture
def raw_churn_df():
    """Matches your actual CSV schema with specific edge cases."""
    return pd.DataFrame({
        'customerID': [1, 2, 3],
        'tenure': [1, 0, 24],
        'TotalCharges': ['29.85', ' ', '1889.5'], # Test: space handling
        'MonthlyCharges': [29.85, 50.0, np.nan],  # Test: numeric imputation
        'InternetService': ['DSL', 'DSL', np.nan], # Test: categorical imputation
        'Churn': ['No', 'Yes', 'No']              # Test: target mapping
    })

@pytest.fixture
def pipeline_params():
    """Provides the config variables as a dictionary for easy function passing."""
    return {
        "target": "Churn",
        "numeric_cols": ['TotalCharges', 'MonthlyCharges'],
        "categorical_cols": ['InternetService'],
        "final_columns": ['customerID', 'tenure', 'TotalCharges', 'MonthlyCharges', 
                         'InternetService','MonthlyChargeRatio', 'churn_binary']
    }

@pytest.fixture
def empty_churn_df():
    """Matches your actual CSV schema with specific edge cases."""
    return pd.DataFrame(columns=['customerID', 'Churn'])

## I/O Handler Tests (`test_io_handler.py`)

This script provides unit tests for the `io_handler.py` module, specifically the `extract_from_csv` and `load_to_csv` functions. The goal is to ensure that reading and writing CSV files works correctly under normal conditions, with empty data, and when files are missing. Each test uses pytest's built-in `tmp_path` fixture to create isolated temporary directories, guaranteeing a clean file system for every test run.

- **Individual Test Cases:**
    - **`test_extract_returns_dataframe`:**
        - **Objective:** Verifies that `extract_from_csv` always returns a pandas DataFrame, regardless of content.
        - **Assertions:**
            - `assert isinstance(df, pd.DataFrame)`: Confirms the return type is a DataFrame, which is critical because all downstream preprocessing functions depend on this contract.

    - **`test_extract_local_file_success`:**
        - **Objective:** Verifies that a valid CSV file is read correctly with the expected number of rows and columns.
        - **Assertions:**
            - `assert len(df) == 3`: Checks that all 3 rows from the fixture data are loaded.
            - `assert 'customerID' in df.columns`: Confirms that expected columns are present in the loaded DataFrame.

    - **`test_extract_from_csv_not_found`:**
        - **Objective:** Verifies that passing a non-existent file path raises a `FileNotFoundError`.
        - **Assertions:**
            - `pytest.raises(FileNotFoundError)`: Ensures the function raises the correct exception type rather than failing silently or returning `None`.

    - **`test_load_to_csv_success`:**
        - **Objective:** Verifies that `load_to_csv` writes a DataFrame to disk, including creating any necessary parent directories.
        - **Assertions:**
            - `assert os.path.exists(output_path)`: Confirms the file was actually created on disk.
            - `assert check_df.iloc[0]["customerID"] == 1`: Reads the file back and verifies the content was not corrupted during the write operation.

    - **`test_extract_empty_dataset`:**
        - **Objective:** Verifies that the function handles an empty CSV (headers only, zero rows) without crashing.
        - **Assertions:**
            - `assert isinstance(df, pd.DataFrame)`: Confirms a DataFrame is still returned.
            - `assert len(df) == 0`: Checks that no phantom rows are created.
            - `assert 'customerID' in df.columns`: Ensures column headers are preserved even when no data rows exist.


In [ ]:
# tests/test_io_handler.py
import pytest
import os
import sys
import pandas as pd
#from unittest.mock import patch

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))
from src.io_handler import extract_from_csv, load_to_csv


def test_extract_returns_dataframe(tmp_path, raw_churn_df):
    """
    Scenario: Ensure the reader always returns a pandas DataFrame.
    Goal: Data integrity. Downstream functions (preprocessing) expect a DataFrame.
    """
    # Arrange

    
    # Act
    
    # Assert

def test_extract_local_file_success(tmp_path, raw_churn_df):
    
    """
    Scenario: User provides a valid local path.
    Goal: Ensure the function reads a standard CSV correctly.
    """
    # Arrange: Create a temporary CSV file
    
    # Act
    
    # Assert
    ##Same number of rows as the input

    ##Testing if some of the columns are present:

def test_extract_from_csv_not_found():
    """
    Scenario: User provides a path that does not exist.
    Goal: Verify the custom error message and logging trigger.
    """ 
    with pytest.raises(FileNotFoundError):
        extract_from_csv("non_existent_path.csv")

def test_load_to_csv_success(tmp_path, raw_churn_df):
    """
    Scenario: Saving a processed DataFrame to a local path.
    Goal: Ensure 'to_csv' is executed and the file actually appears on disk.
    """
    # Arrange: Setup target path
    
    # Act: Attempt to save
    
    # Assert: Check if file exists and content is correct
    # Verification: Read it back to ensure it wasn't corrupted during write


##Empty dataset
def test_extract_empty_dataset(tmp_path, empty_churn_df):
    """
    Scenario: The CSV file exists but is empty (only headers or totally empty).
    Goal: Ensure the script handles empty sources without crashing.
    """
    # Arrange: Create a CSV with only headers
    
    #pd.DataFrame(columns=['customerID', 'Churn']).to_csv(file_path, index=False)
    
    # Act
    
    # Assert


## Preprocessing Logic Tests (`test_preprocessing.py`)

This script provides unit tests for each transformation function in `preprocessing.py`, as well as a full pipeline smoke test. Each test isolates a specific preprocessing step, type conversion, imputation, feature engineering, and target encoding, using the shared `raw_churn_df` fixture from `conftest.py`. This ensures that each function behaves correctly in isolation before testing them together as an integrated chain.

- **Individual Test Cases:**
    - **`test_handle_types_with_fixture`:**
        - **Objective:** Verifies that `handle_types` converts the `TotalCharges` column from string to numeric, correctly handling the space character `' '` as `NaN`.
        - **Assertions:**
            - `assert pd.api.types.is_numeric_dtype(df['TotalCharges'])`: Confirms the column dtype has been converted to a numeric type.
            - `assert np.isnan(df['TotalCharges'][1])`: Verifies that the space value in row 2 was correctly coerced to `NaN` rather than raising an error.

    - **`test_impute_values_with_fixture`:**
        - **Objective:** Verifies that `impute_values` fills missing numeric values with the column mean and missing categorical values with the column mode.
        - **Assertions:**
            - `assert df['MonthlyCharges'][2] > 20`: Confirms the `NaN` in `MonthlyCharges` was replaced with a reasonable mean value derived from the non-null entries.
            - `assert df['InternetService'][2] == 'DSL'`: Confirms the `NaN` in `InternetService` was replaced with the mode (`'DSL'`), which is the most frequent value.

    - **`test_add_features_with_fixture`:**
        - **Objective:** Verifies that `add_features` correctly engineers the `MonthlyChargeRatio` column using the formula `TotalCharges / (tenure + 1)`.
        - **Assertions:**
            - `assert df['MonthlyChargeRatio'][0] == pytest.approx(14.925)`: For row 0 (tenure=1, TotalCharges=29.85), confirms the calculated ratio is `29.85 / 2 = 14.925`, using `pytest.approx` to handle floating-point precision.

    - **`test_format_target_logic`:**
        - **Objective:** Verifies that `format_target` maps the string target column (`'Yes'`/`'No'`) to a new integer column `churn_binary` (`1`/`0`).
        - **Assertions:**
            - `assert 'churn_binary' in result.columns`: Confirms the new column was created.
            - `assert result['churn_binary'].tolist() == [0, 1, 0]`: Verifies the exact mapping for all three rows (`No→0`, `Yes→1`, `No→0`).
            - `assert pd.api.types.is_integer_dtype(result['churn_binary'])`: Confirms the new column has an integer dtype suitable for modelling.

    - **`test_full_pipeline_flow`:**
        - **Objective:** A smoke test that runs all preprocessing steps together via the `clean_and_enrich_churn_data` orchestrator function, verifying that the steps compose correctly end-to-end.
        - **Assertions:**
            - `assert result.shape == (3, 7)`: Confirms the output has the expected number of rows and columns after all transformations.
            - `assert list(result.columns) == pipeline_params["final_columns"]`: Verifies the final column schema matches the expected output specification.
            - `assert result['churn_binary'].iloc[1] == 1`: Spot-checks that the second customer (who churned) has the correct binary target value.


In [ ]:
##Simple test
#test_preprocessing.py

import pytest
import pandas as pd
import numpy as np
import os
import sys

# Add the project root to sys.path to allow imports from src
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))
#Loading the functions to test

"""
# NOTE: No need to import raw_churn_df, pytest finds it in conftest.py automatically!
"""


def test_handle_types_with_fixture(raw_churn_df):
    """
    Scenario: Convert TotalCharges from string to numeric.
    Benefit: Uses the 'TotalCharges' column defined in conftest.py.
    """
    # Act
    
    # Assert
    # TotalCharges should be numeric:

    # The ' ' value in row 2 should have become NaN

def test_impute_values_with_fixture(raw_churn_df, 
                                    pipeline_params):
    """
    Scenario: Fill missing values for both Numeric and Categorical.
    Goal: Verify math and mode logic using the shared dummy data.
    """
    # First, convert types so mean() works
    
    # Act

    
    # Assert
    # Mean of [29.85, 50.0, np.nan] is 26.61

    # Mode of ['DSL', 'DSL', np.nan] is 'DSL'




In [ ]:
##Extended test
#test_preprocessing.py

def test_add_features_with_fixture(raw_churn_df):
    """
    Scenario: Engineering 'MonthlyChargeRatio'.
    Goal: Test formula using the tenure and TotalCharges from dummy data.
    """
    df = handle_types(raw_churn_df)
    
    # Act
    df = add_features(df)
    
    # Assert: For row 0, tenure is 1, TotalCharges is 29.85. Ratio: 29.85 / (1+1) = 14.925
    assert df['MonthlyChargeRatio'][0] == pytest.approx(14.925)



def test_format_target_logic(raw_churn_df, pipeline_params):
    """
    Scenario: Mapping 'Yes' to 1 and 'No' to 0 in the target column.
    Goal: Ensure 'churn_binary' is created with correct integer mapping.
    """
    # Act
    target_col = pipeline_params["target"] # 'Churn'
    result = format_target(raw_churn_df, target_col)
    
    # Assert: Check if the new column exists
    assert 'churn_binary' in result.columns
    
    # Assert: Check the mapping values [No, Yes, No] -> [0, 1, 0]
    expected_values = [0, 1, 0]
    assert result['churn_binary'].tolist() == expected_values
    
    # Assert: Check data type is numeric/int
    assert pd.api.types.is_integer_dtype(result['churn_binary']) or pd.api.types.is_numeric_dtype(result['churn_binary'])
    

def test_full_pipeline_flow(raw_churn_df, 
                            pipeline_params):
    
    #Scenario: High-level Orchestrator test (The 'Smoke Test').
    #Goal: Ensure all steps work together to produce the final schema.
    
    # Act
    result = clean_and_enrich_churn_data(
        raw_churn_df,
        target=pipeline_params["target"],
        numeric_cols=pipeline_params["numeric_cols"],
        categorical_cols=pipeline_params["categorical_cols"],
        final_columns=pipeline_params["final_columns"]
    )
    
    # Assert: Verify the shape and columns
    assert result.shape == (3, 7)
    assert list(result.columns) == pipeline_params["final_columns"]
    assert result['churn_binary'].iloc[1] == 1 # Second customer churned


## Pipeline Integration Tests (`test_pipeline.py`)

This script provides integration tests for the `run_de_pipeline` orchestrator function in `pipeline.py`. Unlike the unit tests in `test_preprocessing.py` which test individual functions in isolation, these tests exercise the full pipeline from file read to file write using real temporary files on disk. This validates that all components, IO handling, preprocessing, and configuration, work together correctly.

- **Individual Test Cases:**
    - **`test_run_de_pipeline_success`:**
        - **Objective:** An end-to-end integration test that writes raw data to a temporary CSV, runs the full pipeline, and verifies the output file is created with the correct schema.
        - **Assertions:**
            - `assert os.path.exists(output_file)`: Confirms the pipeline produced an output file on disk.
            - `assert list(result_df.columns) == pipeline_params["final_columns"]`: Reads the output file back and verifies it contains exactly the expected final columns.
            - `assert len(result_df) == 3`: Confirms no rows were lost or duplicated during processing.

    - **`test_pipeline_failure_propagation_real`:**
        - **Objective:** Verifies that the pipeline correctly propagates a `FileNotFoundError` when given a non-existent input path, without any mocking, testing real failure behaviour.
        - **Assertions:**
            - `pytest.raises(FileNotFoundError)`: Confirms that the error raised by the IO layer bubbles up through the pipeline orchestrator, ensuring failures are not silently swallowed.


In [ ]:
# tests/test_pipeline.py
import pytest
import os
import sys
#from unittest.mock import patch, MagicMock
import pandas as pd

# Path hack
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from src.pipeline import run_de_pipeline

def test_run_de_pipeline_success(tmp_path, 
                                 raw_churn_df, 
                                 pipeline_params):
    """
    Scenario: End-to-End Integration Test using real file system (tmp_path).
    Goal: Ensure the 'orchestrator' successfully moves data from input to output.
    """
    # 1. Arrange: Create a real raw file
    input_file = tmp_path / "raw_churn.csv"
    output_file = tmp_path / "processed_churn.csv"
    raw_churn_df.to_csv(input_file, index=False)
    
    # 2. Act: Run the full pipeline
    run_de_pipeline(
        input_path=str(input_file),
        output_path=str(output_file),
        target=pipeline_params["target"],
        num_cols=pipeline_params["numeric_cols"],
        cat_cols=pipeline_params["categorical_cols"],
        final_cols=pipeline_params["final_columns"]
    )
    
    # 3. Assert: Verify the output file exists and has the final expected columns
    assert os.path.exists(output_file)
    result_df = pd.read_csv(output_file)
    assert list(result_df.columns) == pipeline_params["final_columns"]
    assert len(result_df) == 3




### NOTE: This test is a bit complex, review if we add it

def test_pipeline_failure_propagation_real():
    """
    Scenario: Real-world failure propagation.
    Goal: Verify that if the file is missing, the pipeline raises FileNotFoundError.
    Method: No mocking, just passing a non-existent path.
    """
    # Arrange: A path we know is fake
    fake_path = "data/raw/this_file_does_not_exist_anywhere.csv"
    
    # Act & Assert: Using pytest.raises (Pytest's built-in tool)
    with pytest.raises(FileNotFoundError):
        run_de_pipeline(
            input_path=fake_path,
            output_path="wont_be_created.csv",
            target="Churn",
            num_cols=[],
            cat_cols=[],
            final_cols=[]
        )

## Main Orchestration Tests (`test_main.py`)

This script provides tests for the `main.py` entry point, focusing on orchestration logic and error handling. These tests use `unittest.mock.patch` to replace the actual pipeline execution with mock objects, allowing validation of how `main()` delegates work and handles failures without running the heavyweight pipeline itself.

- **Individual Test Cases:**
    - **`test_main_orchestration_success`:**
        - **Objective:** Verifies that `main()` correctly reads configuration values from `config.py` and passes them as arguments to `run_de_pipeline`.
        - **Assertions:**
            - `mock_pipeline.assert_called_once()`: Confirms that `main()` invoked the pipeline function exactly once.
            - `assert kwargs['input_path'] == config.RAW_DATA_PATH`: Verifies that the input path argument matches the value defined in the config module.
            - `assert kwargs['target'] == config.TARGET` and `assert kwargs['final_cols'] == config.COLUMNS_TO_KEEP`: Confirms that all configuration values are correctly forwarded to the pipeline.

    - **`test_main_error_handling`:**
        - **Objective:** Verifies the `try-except` safety net in `main()`. If the pipeline raises an exception, `main()` should catch it and log the error rather than crashing the entire application.
        - **Assertions:**
            - `mock_logger.error.assert_called_once_with("Pipeline failed: Critical Pipeline Failure")`: Confirms that the `except` block caught the simulated exception and logged it with the expected error message, proving the error handling logic works correctly.

In [ ]:
#test/test_main.py
import pytest
from unittest.mock import patch, MagicMock
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))
from main import main

# @patch acts like a "hijacker". 
# It looks inside 'main.py' and replaces the real 'config' and 
# 'run_de_pipeline' with fake "Mock" objects for the duration of this test.
@patch('main.config')           # Mock object 1: mock_config
@patch('main.run_de_pipeline')  # Mock object 2: mock_pipeline
def test_main_orchestration_success(mock_pipeline, mock_config):
    """
    PURPOSE: Verify that main() is a 'good messenger'.
    Does it correctly grab values from config.py and send them to the pipeline?
    """
    
    # --- 1. ARRANGE (The Setup) ---
    # We tell our 'mock_config' stunt double to pretend it has these specific values.
    # This keeps our test isolated from whatever is actually written in the real config.py.
    mock_config.RAW_DATA_PATH = "data/raw.csv"
    mock_config.PROCESSED_DATA_PATH = "data/output.csv"
    mock_config.TARGET = "Churn"
    mock_config.NUMERIC_COLS = ["total"]
    mock_config.CATEGORICAL_COLS = ["internet"]
    mock_config.COLUMNS_TO_KEEP = ["id", "churn_binary"]

    # --- 2. ACT (The Action) ---
    # We run the main() function. 
    # Because of the @patch decorators above, when main() tries to call 
    # 'run_de_pipeline', it will actually call our 'mock_pipeline' instead.
    main()

    # --- 3. ASSERT (The Verification) ---
    # We check if the 'mock_pipeline' was called exactly once.
    # We also check if the arguments passed to it match our mock_config values.
    # This proves the "wiring" in main.py is correct.
    mock_pipeline.assert_called_once_with(
        input_path="data/raw.csv",
        output_path="data/output.csv",
        target="Churn",
        num_cols=["total"],
        cat_cols=["internet"],
        final_cols=["id", "churn_binary"]
    )


# TEST: ERROR HANDLING

@patch('main.run_de_pipeline')
@patch('main.logger')  # We intercept the 'logger' object inside main.py
def test_main_error_handling(mock_logger, mock_pipeline):
    """
    PURPOSE: Verify the 'Safety Net'.
    If the pipeline crashes, does main() catch the error and log it?
    """
    
    # --- 1. ARRANGE (The Setup) ---
    # We tell the mock_pipeline to "explode" (raise an Exception) when called.
    # This simulates a real-world error like a missing file or network failure.
    mock_pipeline.side_effect = Exception("Connection Timeout")

    # --- 2. ACT (The Action) ---
    # We run main(). 
    # If our try/except block in main.py is working, the test won't crash.
    main()

    # --- 3. ASSERT (The Verification) ---
    # Instead of checking the output, we check the Logger.
    # Did main() call logger.error() with the specific message we expected?
    # This proves that our error handling logic is actually executing.
    mock_logger.error.assert_called_once_with("Pipeline failed: Connection Timeout")